# LeetCode 359 — Logger Rate Limiter bằng Python

**Link đề:** https://leetcode.com/problems/logger-rate-limiter/
**Độ khó:** Easy · **Chủ đề:** Hash Table, Design, Data Stream

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Logger:

    def __init__(self):
        ...

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        ...
```

## Mục tiêu học
1. Làm quen dạng bài **thiết kế (Design)**: viết một **class** có trạng thái, được gọi nhiều lần.
2. Hiểu bài toán **giới hạn tần suất (rate limiting)** trên **luồng dữ liệu (data stream)**.
3. Đi từ cách trực giác (lưu toàn bộ lịch sử) → **hash map** lưu thời điểm in gần nhất `O(1)` mỗi lần gọi.
4. Biết cách **dọn bộ nhớ** bằng hàng đợi (`deque`) + set, hoặc bằng kỹ thuật "hai xô" (two buckets).
5. Trace trạng thái dict sau mỗi lần gọi; viết bộ chạy (driver) mô phỏng cách LeetCode gọi class.

## 1. Đề bài (theo hiểu biết về bài)

Thiết kế một hệ thống ghi log nhận **luồng** các thông điệp kèm **mốc thời gian** (giây). Mỗi thông điệp **giống nhau** chỉ được in **tối đa một lần mỗi 10 giây**: nếu thông điệp được in ở thời điểm `t` thì các thông điệp giống hệt sẽ **không** được in cho tới thời điểm `t + 10` (tại `t + 10` thì được in lại).

Các thông điệp đến theo **thứ tự thời gian không giảm**; nhiều thông điệp có thể đến cùng một mốc thời gian.

Cài đặt class `Logger`:
- `Logger()` — khởi tạo đối tượng.
- `shouldPrintMessage(timestamp, message)` — trả `True` nếu `message` **được in** tại `timestamp`, ngược lại `False`.

**Ví dụ:**

```text
Input:
["Logger", "shouldPrintMessage", "shouldPrintMessage", "shouldPrintMessage",
 "shouldPrintMessage", "shouldPrintMessage", "shouldPrintMessage"]
[[], [1, "foo"], [2, "bar"], [3, "foo"], [8, "bar"], [10, "foo"], [11, "foo"]]
Output:
[null, true, true, false, false, false, true]
```

| Lời gọi | Kết quả | Vì sao |
|---|:---:|---|
| `(1, "foo")` | `True` | lần đầu thấy "foo"; được in tiếp sớm nhất ở 1 + 10 = 11 |
| `(2, "bar")` | `True` | lần đầu thấy "bar"; lần sau sớm nhất ở 12 |
| `(3, "foo")` | `False` | 3 < 11 |
| `(8, "bar")` | `False` | 8 < 12 |
| `(10, "foo")` | `False` | 10 < 11 |
| `(11, "foo")` | `True` | 11 ≥ 11; lần sau sớm nhất ở 21 |

**Constraints (theo đề gốc):** `0 <= timestamp <= 10^9`; timestamp truyền vào theo thứ tự **không giảm**; `1 <= len(message) <= 30`; tối đa `10^4` lời gọi `shouldPrintMessage`.

**Lưu ý quan trọng:** chỉ khi thông điệp **được in** (trả `True`) thì mới tính lại mốc 10 giây. Lần bị từ chối (`False`) **không** làm "gia hạn" thời gian chờ.

## 2. Nền tảng Python cho bài thiết kế

### 2.1. `class`, `__init__`, `self` — đối tượng có "trí nhớ"

Một **class** là bản thiết kế; một **đối tượng (object)** tạo từ class giữ **trạng thái** riêng giữa các lần gọi method.

- `__init__(self)` chạy **một lần** khi tạo đối tượng: `logger = Logger()`.
- `self.x = ...` lưu dữ liệu vào chính đối tượng; method khác đọc lại qua `self.x`.

In [1]:
class Counter2:
    def __init__(self):
        self.count = 0              # trạng thái được giữ lại giữa các lần gọi

    def hit(self) -> int:
        self.count += 1
        return self.count

c1 = Counter2()
c2 = Counter2()
print(c1.hit(), c1.hit(), c1.hit())   # 1 2 3
print(c2.hit())                       # 1 — đối tượng khác, trạng thái riêng

1 2 3
1


### 2.2. `dict` lưu "thông điệp → thời điểm"

Khóa (`key`) là chuỗi `message`, giá trị (`value`) là một số thời gian. `msg in d` kiểm tra tồn tại trung bình `O(1)`, `d.get(msg, default)` đọc an toàn.

In [2]:
last = {}
last["foo"] = 1
last["bar"] = 2
print(last)
print("'foo' in last:", "foo" in last, "| last.get('baz', None):", last.get("baz", None))
last["foo"] = 11                 # ghi đè giá trị cũ
print(last)

{'foo': 1, 'bar': 2}
'foo' in last: True | last.get('baz', None): None
{'foo': 11, 'bar': 2}


### 2.3. Bộ chạy kiểu LeetCode (driver)

LeetCode mô tả test bằng hai list: **tên thao tác** và **tham số**. Ta viết hàm `run_ops` để mô phỏng: thao tác đầu tạo đối tượng (kết quả `None`), các thao tác sau gọi method bằng `getattr(obj, tên)(*tham_số)`.

- `getattr(obj, "shouldPrintMessage")` lấy method theo **tên chuỗi**.
- `f(*args)` "rải" list `args` thành các tham số rời: `f(*[1, "foo"])` ≡ `f(1, "foo")`.

In [3]:
def run_ops(cls, ops, args):
    out = []
    obj = None
    for op, a in zip(ops, args):          # zip ghép từng cặp (op, a)
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print(list(zip(["x", "y"], [[1], [2]])))

OPS = ["Logger", "shouldPrintMessage", "shouldPrintMessage", "shouldPrintMessage",
       "shouldPrintMessage", "shouldPrintMessage", "shouldPrintMessage"]
ARGS = [[], [1, "foo"], [2, "bar"], [3, "foo"], [8, "bar"], [10, "foo"], [11, "foo"]]
EXPECTED = [None, True, True, False, False, False, True]

[('x', [1]), ('y', [2])]


## 3. Cách 1 — Trực giác: lưu toàn bộ lịch sử các lần in

Người mới hay nghĩ: "Giữ một list các cặp `(thời_điểm, thông_điệp)` **đã in**. Khi có thông điệp mới, quét ngược lịch sử xem có lần in nào của cùng thông điệp trong 10 giây gần đây không."

In [4]:
class LoggerHistory:
    def __init__(self):
        self.printed = []                       # list các (timestamp, message) đã in

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        for t, m in self.printed:               # quét toàn bộ lịch sử: O(số lần đã in)
            if m == message and timestamp - t < 10:
                return False
        self.printed.append((timestamp, message))
        return True

print(run_ops(LoggerHistory, OPS, ARGS))
print("Khớp expected:", run_ops(LoggerHistory, OPS, ARGS) == EXPECTED)

[None, True, True, False, False, False, True]
Khớp expected: True


**Tuple** `(timestamp, message)` là một bộ giá trị cố định, không sửa được; `for t, m in ...` **mở gói (unpack)** từng tuple thành hai biến.

### 3.1. Nút thắt

- Mỗi lần gọi quét toàn bộ lịch sử → với `10^4` lời gọi, tổng có thể tới `~5·10^7` phép so sánh.
- Bộ nhớ lịch sử tăng mãi.

**Insight:** với mỗi thông điệp, ta **chỉ cần nhớ lần in gần nhất** (hoặc thời điểm sớm nhất được in tiếp). Những lần in cũ hơn không bao giờ ảnh hưởng quyết định nữa. → Dùng **dict** `message → mốc thời gian`.

In [5]:
# Minh họa: chỉ lần in gần nhất quan trọng
history_foo = [1, 11, 21]          # các lần 'foo' đã được in
for t_now in [25, 31]:
    gaps = [t_now - t for t in history_foo]
    print(f"t_now={t_now}: khoảng cách tới các lần in = {gaps} -> in được? {gaps[-1] >= 10} (chỉ cần nhìn khoảng cách nhỏ nhất = lần gần nhất)")

t_now=25: khoảng cách tới các lần in = [24, 14, 4] -> in được? False (chỉ cần nhìn khoảng cách nhỏ nhất = lần gần nhất)
t_now=31: khoảng cách tới các lần in = [30, 20, 10] -> in được? True (chỉ cần nhìn khoảng cách nhỏ nhất = lần gần nhất)


## 4. Cách 2 — Hash map lưu thời điểm **được in tiếp sớm nhất** (lời giải chính)

Ba đoạn nhỏ:

**Đoạn 1 — trạng thái:** `self.next_allowed = {}` lưu `message → thời điểm sớm nhất được in lần tới`.

**Đoạn 2 — quyết định:** nếu `timestamp < self.next_allowed.get(message, 0)` → còn trong thời gian chờ → `False`.

**Đoạn 3 — cập nhật khi in:** ghi `self.next_allowed[message] = timestamp + 10` và trả `True`.

Vì sao mặc định `0`? Thông điệp chưa từng thấy → mọi `timestamp >= 0` đều được in.

In [6]:
class LoggerNextAllowed:
    def __init__(self):
        self.next_allowed = {}          # message -> mốc sớm nhất được in lại

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if timestamp < self.next_allowed.get(message, 0):
            return False                # còn trong 10 giây chờ, KHÔNG cập nhật
        self.next_allowed[message] = timestamp + 10
        return True

print(run_ops(LoggerNextAllowed, OPS, ARGS))

[None, True, True, False, False, False, True]


### 4.1. Cách viết tương đương: lưu **lần in gần nhất**

Lưu `last[message] = timestamp` khi in; điều kiện in là "chưa từng in" **hoặc** `timestamp - last[message] >= 10`. Hai cách tương đương, chỉ khác chỗ cộng 10.

In [7]:
class LoggerLast:
    def __init__(self):
        self.last = {}

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if message in self.last and timestamp - self.last[message] < 10:
            return False
        self.last[message] = timestamp
        return True

print(run_ops(LoggerLast, OPS, ARGS) == EXPECTED)

True


## 5. TRACE từng bước — Cách 2 trên ví dụ của đề

| Lời gọi | `next_allowed` trước | so sánh | Kết quả | `next_allowed` sau |
|---|---|---|:---:|---|
| `(1, foo)` | `{}` | 1 < 0? không | `True` | `{foo: 11}` |
| `(2, bar)` | `{foo: 11}` | 2 < 0? không | `True` | `{foo: 11, bar: 12}` |
| `(3, foo)` | `{foo: 11, bar: 12}` | 3 < 11? có | `False` | không đổi |
| `(8, bar)` | — | 8 < 12? có | `False` | không đổi |
| `(10, foo)` | — | 10 < 11? có | `False` | không đổi |
| `(11, foo)` | `{foo: 11, bar: 12}` | 11 < 11? **không** | `True` | `{foo: 21, bar: 12}` |

Nhánh đáng chú ý: `(10, foo)` — chỉ thiếu 1 giây; `(11, foo)` — đúng biên `t + 10` thì được in.

In [8]:
class LoggerTrace(LoggerNextAllowed):          # kế thừa: dùng lại __init__ của lớp cha
    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        before = dict(self.next_allowed)         # bản sao để in trạng thái trước
        limit = before.get(message, 0)
        ok = super().shouldPrintMessage(timestamp, message)
        print(f"({timestamp:>2}, {message!r:<6}) | trước={before} | {timestamp} < {limit}? {timestamp < limit}"
              f" -> {ok} | sau={self.next_allowed}")
        return ok

result = run_ops(LoggerTrace, OPS, ARGS)
print("Output:", result)
assert result == EXPECTED

( 1, 'foo' ) | trước={} | 1 < 0? False -> True | sau={'foo': 11}
( 2, 'bar' ) | trước={'foo': 11} | 2 < 0? False -> True | sau={'foo': 11, 'bar': 12}
( 3, 'foo' ) | trước={'foo': 11, 'bar': 12} | 3 < 11? True -> False | sau={'foo': 11, 'bar': 12}
( 8, 'bar' ) | trước={'foo': 11, 'bar': 12} | 8 < 12? True -> False | sau={'foo': 11, 'bar': 12}
(10, 'foo' ) | trước={'foo': 11, 'bar': 12} | 10 < 11? True -> False | sau={'foo': 11, 'bar': 12}
(11, 'foo' ) | trước={'foo': 11, 'bar': 12} | 11 < 11? False -> True | sau={'foo': 21, 'bar': 12}
Output: [None, True, True, False, False, False, True]


Giải thích vài cú pháp mới trong cell trên:
- `class LoggerTrace(LoggerNextAllowed)` — **kế thừa**: lớp con có sẵn mọi method của lớp cha.
- `super().shouldPrintMessage(...)` — gọi phiên bản của **lớp cha**, để logic trace và logic thật là một.
- `dict(self.next_allowed)` — tạo **bản sao** để in trạng thái "trước" (nếu không sao chép, ta chỉ in cùng một object đã bị sửa).
- `{message!r}` trong f-string — in dạng `repr`, tức có dấu nháy: `'foo'`. `:<6` căn trái độ rộng 6.

## 6. Cách 3 — Dọn bộ nhớ bằng hàng đợi (`deque`) + set

Cách 2 giữ mọi thông điệp **mãi mãi** trong dict. Nếu luồng có hàng triệu thông điệp khác nhau, bộ nhớ phình to. Ý tưởng **cửa sổ trượt 10 giây**:

- `queue`: các `(timestamp, message)` **đã in** trong 10 giây gần nhất, theo thứ tự thời gian.
- `active`: set các thông điệp đang nằm trong `queue`.
- Mỗi lần gọi: **bỏ khỏi đầu hàng** mọi bản ghi có `timestamp - t >= 10` (đã hết hạn), xóa chúng khỏi `active`. Sau đó: nếu `message in active` → `False`; ngược lại thêm vào và trả `True`.

Bộ nhớ chỉ còn tỉ lệ với số thông điệp được in trong **10 giây gần nhất**.

**`collections.deque`** là hàng đợi hai đầu: `append` (thêm cuối), `popleft` (lấy đầu) đều `O(1)`. (`list.pop(0)` thì `O(n)`.)

In [9]:
from collections import deque
q = deque()
q.append((1, "foo"))
q.append((2, "bar"))
print("q =", q, "| q[0] =", q[0])
print("popleft ->", q.popleft(), "| còn:", q)

q = deque([(1, 'foo'), (2, 'bar')]) | q[0] = (1, 'foo')
popleft -> (1, 'foo') | còn: deque([(2, 'bar')])


In [10]:
class LoggerQueue:
    def __init__(self):
        self.queue = deque()          # (timestamp, message) đã in, tăng dần theo thời gian
        self.active = set()           # message đang bị khóa

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        while self.queue and timestamp - self.queue[0][0] >= 10:
            _, old_msg = self.queue.popleft()      # hết hạn -> mở khóa
            self.active.discard(old_msg)
        if message in self.active:
            return False
        self.queue.append((timestamp, message))
        self.active.add(message)
        return True

print(run_ops(LoggerQueue, OPS, ARGS) == EXPECTED)

True


**Vì sao đúng?** Mỗi thông điệp chỉ có tối đa **một** bản ghi trong `queue` (vì khi đang `active` thì không được thêm). Bản ghi đó bị bỏ đúng khi đã qua đủ 10 giây. Vì timestamp **không giảm**, các bản ghi hết hạn luôn nằm **ở đầu** hàng → chỉ cần nhìn `queue[0]`.

`_` là tên biến quy ước cho giá trị **không dùng tới**.

In [11]:
def trace_queue(ops, args):
    obj = LoggerQueue()
    for a in args[1:]:
        ok = obj.shouldPrintMessage(*a)
        print(f"{str(a):<12} -> {str(ok):<5} | queue={list(obj.queue)} | active={sorted(obj.active)}")

trace_queue(OPS, ARGS)

[1, 'foo']   -> True  | queue=[(1, 'foo')] | active=['foo']
[2, 'bar']   -> True  | queue=[(1, 'foo'), (2, 'bar')] | active=['bar', 'foo']
[3, 'foo']   -> False | queue=[(1, 'foo'), (2, 'bar')] | active=['bar', 'foo']
[8, 'bar']   -> False | queue=[(1, 'foo'), (2, 'bar')] | active=['bar', 'foo']
[10, 'foo']  -> False | queue=[(1, 'foo'), (2, 'bar')] | active=['bar', 'foo']
[11, 'foo']  -> True  | queue=[(2, 'bar'), (11, 'foo')] | active=['bar', 'foo']


## 7. Cách 4 — "Hai xô" (two buckets): dọn bộ nhớ mà không cần hàng đợi

Chia thời gian thành các **khung 10 giây** theo mốc `self.start`. Giữ hai dict:
- `self.cur`: thông điệp **in** trong khung hiện tại `[start, start + 10)`, giá trị là thời điểm in.
- `self.prev`: của khung liền trước.

Khi `timestamp >= start + 20` → cả hai khung cũ đều đã quá 10 giây → xóa cả hai. Khi `timestamp >= start + 10` → khung hiện tại thành khung trước. Một thông điệp in quá 10 giây trước chắc chắn không nằm ở hai khung này, nên chỉ cần tra `cur` rồi `prev`.

In [12]:
class LoggerBuckets:
    def __init__(self):
        self.start = 0
        self.cur = {}
        self.prev = {}

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if timestamp >= self.start + 20:          # quá xa: bỏ hết
            self.prev, self.cur = {}, {}
            self.start = timestamp
        elif timestamp >= self.start + 10:        # sang khung mới
            self.prev, self.cur = self.cur, {}
            self.start += 10
        last = self.cur.get(message, self.prev.get(message))
        if last is not None and timestamp - last < 10:
            return False
        self.cur[message] = timestamp
        return True

print(run_ops(LoggerBuckets, OPS, ARGS) == EXPECTED)

True


**`None` và `is None`:** `None` là giá trị "không có gì". So sánh với `None` nên dùng `is` / `is not`. `self.cur.get(message, self.prev.get(message))` đọc khung hiện tại, thiếu thì đọc khung trước, thiếu nữa thì `None`.

Lưu ý tinh tế: `self.start` luôn tăng đúng bội 10 trong nhánh `elif`, nên mỗi khung dài đúng 10 giây; nhánh `if` "nhảy cóc" khi có khoảng lặng dài.

## 8. Code nộp LeetCode

In [13]:
class Logger:

    def __init__(self):
        self.next_allowed = {}

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if timestamp < self.next_allowed.get(message, 0):
            return False
        self.next_allowed[message] = timestamp + 10
        return True


# Your Logger object will be instantiated and called as such:
logger = Logger()
print([logger.shouldPrintMessage(t, m) for t, m in [(1, "foo"), (2, "bar"), (3, "foo"), (8, "bar"), (10, "foo"), (11, "foo")]])

[True, True, False, False, False, True]


## 9. Vì sao đúng?

**Invariant (Cách 2):** *với mỗi thông điệp `m` đã từng được in, `next_allowed[m] = t_last + 10`, trong đó `t_last` là lần **in** gần nhất của `m`; thông điệp chưa in thì không có trong dict.*

- Khởi tạo: dict rỗng — đúng (chưa in gì).
- Gọi `(t, m)`:
  - Nếu `t < next_allowed[m]` ⇔ `t - t_last < 10` → theo đề phải từ chối; ta trả `False` và **không** sửa dict → invariant giữ.
  - Ngược lại (chưa in bao giờ, hoặc đã đủ 10 giây) → đề cho phép in; ta trả `True` và cập nhật `t_last = t` → invariant giữ.

Vì chỉ lần in **gần nhất** quyết định (các lần cũ hơn có `t_old ≤ t_last` nên `t - t_old ≥ t - t_last`), nhớ một giá trị là đủ.

## 10. Độ phức tạp

Gọi `q` là số lời gọi, `u` là số thông điệp khác nhau, `L ≤ 30` là độ dài thông điệp (băm chuỗi tốn `O(L)`).

| Cách | Time mỗi lời gọi | Space |
|---|---|---|
| 1. Lịch sử | `O(số lần đã in · L)` | `O(q)` |
| 2. Dict next_allowed / last | `O(L)` ≈ `O(1)` trung bình | `O(u)` — không bao giờ giảm |
| 3. Deque + set | `O(L)` **phân bổ (amortized)** — mỗi bản ghi vào/ra hàng đúng 1 lần | `O(số thông điệp in trong 10 giây gần nhất)` |
| 4. Hai xô | `O(L)` trung bình | `O(số thông điệp in trong ≤ 20 giây gần nhất)` |

"Phân bổ" nghĩa là: một lời gọi có thể pop nhiều phần tử, nhưng **tổng** số pop trên toàn bộ `q` lời gọi không vượt quá tổng số append, nên trung bình mỗi lời gọi `O(1)`.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu 5 cài đặt

In [14]:
impls = [LoggerHistory, LoggerNextAllowed, LoggerLast, LoggerQueue, LoggerBuckets, Logger]

def calls_to_ops(calls):
    return ["Logger"] + ["shouldPrintMessage"] * len(calls), [[]] + [list(c) for c in calls]

tests = [
    ("mẫu đề", [(1, "foo"), (2, "bar"), (3, "foo"), (8, "bar"), (10, "foo"), (11, "foo")],
     [True, True, False, False, False, True]),
    ("cùng timestamp", [(0, "a"), (0, "a"), (0, "b")], [True, False, True]),
    ("đúng biên +10", [(5, "x"), (15, "x"), (24, "x"), (25, "x")], [True, True, False, True]),
    ("bị từ chối không gia hạn", [(0, "x"), (9, "x"), (10, "x")], [True, False, True]),
    ("timestamp 0 và rất lớn", [(0, "z"), (10**9, "z")], [True, True]),
    ("khoảng lặng dài", [(1, "a"), (2, "b"), (100, "a"), (105, "b"), (109, "a"), (110, "a")],
     [True, True, True, True, False, True]),
]
for name, calls, exp in tests:
    ops, args = calls_to_ops(calls)
    for cls in impls:
        got = run_ops(cls, ops, args)[1:]
        assert got == exp, (name, cls.__name__, got, exp)
    print(f"{name:<26} PASS -> {exp}")

mẫu đề                     PASS -> [True, True, False, False, False, True]
cùng timestamp             PASS -> [True, False, True]
đúng biên +10              PASS -> [True, True, False, True]
bị từ chối không gia hạn   PASS -> [True, False, True]
timestamp 0 và rất lớn     PASS -> [True, True]
khoảng lặng dài            PASS -> [True, True, True, True, False, True]


In [15]:
# Kiểm thử ngẫu nhiên có seed: mọi cài đặt phải giống LoggerHistory (oracle)
import random
rng = random.Random(359)
for _ in range(300):
    t = 0
    calls = []
    for _ in range(rng.randint(1, 40)):
        t += rng.choice([0, 0, 1, 2, 3, 5, 9, 10, 11, 25])
        calls.append((t, rng.choice("abc")))
    ops, args = calls_to_ops(calls)
    expected = run_ops(LoggerHistory, ops, args)
    for cls in impls:
        assert run_ops(cls, ops, args) == expected, (cls.__name__, calls)
print("300 luồng ngẫu nhiên PASS cho", len(impls), "cài đặt")

300 luồng ngẫu nhiên PASS cho 6 cài đặt


In [16]:
# So sánh bộ nhớ: dict của Cách 2 phình to, deque của Cách 3 thì không
a2, a3 = LoggerNextAllowed(), LoggerQueue()
for t in range(10000):
    msg = f"msg{t}"                 # mỗi giây một thông điệp mới
    a2.shouldPrintMessage(t, msg)
    a3.shouldPrintMessage(t, msg)
print("Cách 2 giữ", len(a2.next_allowed), "khóa | Cách 3 giữ", len(a3.queue), "bản ghi")

Cách 2 giữ 10000 khóa | Cách 3 giữ 10 bản ghi


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Cập nhật thời gian cả khi **từ chối**

```python
ok = timestamp - self.last.get(message, -10) >= 10
self.last[message] = timestamp     # SAI: ghi cả khi ok = False
return ok
```
Vì sao sai: `(0, x) True`, `(9, x) False` nhưng lại ghi `last = 9` → `(10, x)` bị từ chối sai (đúng phải `True`).

### Lỗi 2 — Dùng `<=` thay `<` ở biên

`timestamp - last <= 10` → `False` → chặn cả lúc đúng `t + 10`, trong khi đề cho in lại tại `t + 10`.

### Lỗi 3 — Mặc định sai cho thông điệp mới

`self.last.get(message, 0)` rồi kiểm tra `timestamp - 0 >= 10` → thông điệp mới ở `timestamp = 3` bị từ chối sai. Mặc định phải là `-10` (hoặc dùng `message in self.last`).

### Lỗi 4 — Biến lớp thay vì biến đối tượng

Khai báo `memo = {}` ngay trong thân class (không qua `self` trong `__init__`) → **mọi** đối tượng dùng chung một dict; test case sau bị ảnh hưởng bởi test case trước.

In [17]:
class WrongUpdateAlways:
    def __init__(self):
        self.last = {}
    def shouldPrintMessage(self, timestamp, message):
        ok = timestamp - self.last.get(message, -10) >= 10
        self.last[message] = timestamp
        return ok

class WrongBoundary:
    def __init__(self):
        self.last = {}
    def shouldPrintMessage(self, timestamp, message):
        if message in self.last and timestamp - self.last[message] <= 10:
            return False
        self.last[message] = timestamp
        return True

class WrongDefault:
    def __init__(self):
        self.last = {}
    def shouldPrintMessage(self, timestamp, message):
        if timestamp - self.last.get(message, 0) < 10:
            return False
        self.last[message] = timestamp
        return True

class WrongShared:
    last = {}                                   # biến của LỚP, dùng chung!
    def shouldPrintMessage(self, timestamp, message):
        if message in self.last and timestamp - self.last[message] < 10:
            return False
        self.last[message] = timestamp
        return True

for cls, calls in [(WrongUpdateAlways, [(0, "x"), (9, "x"), (10, "x")]),
                   (WrongBoundary, [(5, "x"), (15, "x")]),
                   (WrongDefault, [(3, "x")])]:
    o, r = cls(), Logger()
    print(f"{cls.__name__:<18} sai: {[o.shouldPrintMessage(*c) for c in calls]} | đúng: {[r.shouldPrintMessage(*c) for c in calls]}")

w1, w2 = WrongShared(), WrongShared()
print("WrongShared: đối tượng 1 ->", w1.shouldPrintMessage(0, "x"), "| đối tượng MỚI 2 ->", w2.shouldPrintMessage(0, "x"), "(đúng phải True)")

WrongUpdateAlways  sai: [True, False, False] | đúng: [True, False, True]
WrongBoundary      sai: [True, False] | đúng: [True, True]
WrongDefault       sai: [False] | đúng: [True]
WrongShared: đối tượng 1 -> True | đối tượng MỚI 2 -> False (đúng phải True)


## 13. Mẹo nhận dạng pattern

- "Mỗi khóa chỉ được ... một lần trong **K giây/khoảng**" → **dict khóa → mốc thời gian gần nhất**.
- "Luồng dữ liệu theo thời gian không giảm" + "hết hạn sau K" → **hàng đợi (deque)** làm cửa sổ trượt; phần tử hết hạn luôn ở đầu.
- Bài **Design**: xác định (1) trạng thái cần nhớ tối thiểu, (2) mỗi thao tác đọc/sửa trạng thái thế nào, (3) độ phức tạp mỗi thao tác.
- Bài anh em: **362 Design Hit Counter** (đếm số lần trong 300 giây gần nhất), **346 Moving Average from Data Stream** (cửa sổ trượt cố định), **933 Number of Recent Calls**.

## 14. Biến thể

### 14.1. Giới hạn **mỗi thông điệp tối đa `k` lần trong `w` giây**

Mỗi thông điệp giữ một `deque` các thời điểm in trong cửa sổ; nếu đã đủ `k` thì từ chối.

In [18]:
from collections import defaultdict

class RateLimiterK:
    def __init__(self, k: int, w: int):
        self.k, self.w = k, w
        self.times = defaultdict(deque)     # message -> deque các lần in còn trong cửa sổ

    def allow(self, timestamp: int, message: str) -> bool:
        q = self.times[message]
        while q and timestamp - q[0] >= self.w:
            q.popleft()
        if len(q) >= self.k:
            return False
        q.append(timestamp)
        return True

rl = RateLimiterK(k=2, w=10)
print([rl.allow(t, "x") for t in [0, 1, 2, 9, 10, 11, 12]])
# 0 ✔, 1 ✔, 2 ✘ (đã 2 lần), 9 ✘, 10 ✔ (lần ở 0 hết hạn), 11 ✔ (lần ở 1 hết hạn), 12 ✘

[True, True, False, False, True, True, False]


`defaultdict(deque)` là dict mà khi đọc khóa **chưa có**, tự tạo giá trị mặc định bằng cách gọi `deque()` — không cần `if message not in ...`.

### 14.2. Design Hit Counter (ý tưởng LeetCode 362)

Đếm số "hit" trong 300 giây gần nhất: deque các timestamp, bỏ đầu hàng khi `timestamp - q[0] >= 300`, trả `len(q)`.

In [19]:
class HitCounter:
    def __init__(self):
        self.q = deque()
    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)
    def getHits(self, timestamp: int) -> int:
        while self.q and timestamp - self.q[0] >= 300:
            self.q.popleft()
        return len(self.q)

hc = HitCounter()
for t in [1, 2, 3]:
    hc.hit(t)
print(hc.getHits(4), end=" ")
hc.hit(300)
print(hc.getHits(300), hc.getHits(301))   # 3 4 3

3 4 3


## 15. Bài tập tự luyện (TODO)

**Bài 1 (dự đoán output).** Với `Logger` chuẩn, chuỗi lời gọi `(0,"a"), (5,"b"), (9,"a"), (10,"a"), (14,"b"), (15,"b")` trả về gì?

**Bài 2 (tính tay).** Lập bảng `next_allowed` sau mỗi lời gọi ở Bài 1.

**Bài 3 (điền chỗ trống).** Hoàn thiện `ExerciseLogger` theo kiểu lưu **lần in gần nhất**.

**Bài 4 (đổi tham số).** Viết `LoggerW(w)` cho phép chọn độ dài khoảng chờ `w` thay vì 10.

**Bài 5 (biến thể).** Viết `LoggerQueue` của riêng bạn (deque + set) và đếm số phần tử còn lại sau luồng 1000 thông điệp khác nhau.

**Bài 6 (mini challenge).** Viết `UserRateLimiter`: mỗi **user** được tối đa 3 request trong 60 giây; method `allow(timestamp, user) -> bool`.

In [20]:
class ExerciseLogger:
    def __init__(self):
        # TODO: tạo dict self.last
        pass

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        # TODO 1: nếu message đã có và timestamp - last < 10 -> False
        # TODO 2: ngược lại cập nhật last và trả True
        pass

# Tự làm xong rồi mới mở test:
# assert run_ops(ExerciseLogger, OPS, ARGS) == EXPECTED

In [21]:
class LoggerW:
    def __init__(self, w: int):
        # TODO
        pass

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        # TODO
        pass

# lw = LoggerW(3)
# assert [lw.shouldPrintMessage(t, "m") for t in [0, 2, 3, 5, 6]] == [True, False, True, False, True]

In [22]:
class ExerciseLoggerQueue:
    def __init__(self):
        # TODO: deque + set
        pass

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        # TODO
        pass

# assert run_ops(ExerciseLoggerQueue, OPS, ARGS) == EXPECTED

In [23]:
class UserRateLimiter:
    def __init__(self, k: int = 3, w: int = 60):
        # TODO
        pass

    def allow(self, timestamp: int, user: str) -> bool:
        # TODO
        pass

# u = UserRateLimiter()
# assert [u.allow(t, "an") for t in [0, 10, 20, 30, 60, 61]] == [True, True, True, False, True, False]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1–2 | mỗi thông điệp có "đồng hồ" riêng | cập nhật chỉ khi `True` | `a: 0→10, b: 5→15` |
| 3 | nhớ lần in gần nhất | `message in self.last` | `if ... and timestamp - self.last[message] < 10: return False` |
| 4 | thay hằng số 10 | lưu `self.w` trong `__init__` | `self.next_allowed[m] = timestamp + self.w` |
| 5 | bỏ phần tử hết hạn ở đầu hàng | `while q and t - q[0][0] >= 10` | `popleft` + `discard` |
| 6 | giống `RateLimiterK` | mỗi user một deque | `defaultdict(deque)`, `len(q) >= k` thì `False` |

## 17. Đáp án tham khảo

**Bài 1:** `[True, True, False, True, False, True]`.

**Bài 2:** `(0,a)` → `{a:10}`; `(5,b)` → `{a:10, b:15}`; `(9,a)` 9<10 từ chối; `(10,a)` → `{a:20, b:15}`; `(14,b)` 14<15 từ chối; `(15,b)` → `{a:20, b:25}`.

In [24]:
class AnswerLogger:
    def __init__(self):
        self.last = {}
    def shouldPrintMessage(self, timestamp, message):
        if message in self.last and timestamp - self.last[message] < 10:
            return False
        self.last[message] = timestamp
        return True

class AnswerLoggerW:
    def __init__(self, w):
        self.w = w
        self.next_allowed = {}
    def shouldPrintMessage(self, timestamp, message):
        if timestamp < self.next_allowed.get(message, 0):
            return False
        self.next_allowed[message] = timestamp + self.w
        return True

class AnswerLoggerQueue:
    def __init__(self):
        self.q, self.active = deque(), set()
    def shouldPrintMessage(self, timestamp, message):
        while self.q and timestamp - self.q[0][0] >= 10:
            self.active.discard(self.q.popleft()[1])
        if message in self.active:
            return False
        self.q.append((timestamp, message)); self.active.add(message)
        return True

class AnswerUserRateLimiter:
    def __init__(self, k=3, w=60):
        self.k, self.w = k, w
        self.times = defaultdict(deque)
    def allow(self, timestamp, user):
        q = self.times[user]
        while q and timestamp - q[0] >= self.w:
            q.popleft()
        if len(q) >= self.k:
            return False
        q.append(timestamp)
        return True

lg = Logger()
calls = [(0, "a"), (5, "b"), (9, "a"), (10, "a"), (14, "b"), (15, "b")]
ans1 = [lg.shouldPrintMessage(*c) for c in calls]
print("Bài 1:", ans1, "| next_allowed cuối:", lg.next_allowed)
assert ans1 == [True, True, False, True, False, True]
assert run_ops(AnswerLogger, OPS, ARGS) == EXPECTED
lw = AnswerLoggerW(3)
assert [lw.shouldPrintMessage(t, "m") for t in [0, 2, 3, 5, 6]] == [True, False, True, False, True]
assert run_ops(AnswerLoggerQueue, OPS, ARGS) == EXPECTED
aq = AnswerLoggerQueue()
for t in range(1000):
    aq.shouldPrintMessage(t, f"m{t}")
print("Bài 5: còn", len(aq.q), "bản ghi trong hàng đợi")
u = AnswerUserRateLimiter()
assert [u.allow(t, "an") for t in [0, 10, 20, 30, 60, 61]] == [True, True, True, False, True, False]
print("Đáp án PASS toàn bộ!")

Bài 1: [True, True, False, True, False, True] | next_allowed cuối: {'a': 20, 'b': 25}
Bài 5: còn 10 bản ghi trong hàng đợi
Đáp án PASS toàn bộ!


## 18. Template nhớ nhanh + Checklist submit

```python
class Logger:
    def __init__(self):
        self.next_allowed = {}                    # message -> mốc sớm nhất được in lại

    def shouldPrintMessage(self, timestamp: int, message: str) -> bool:
        if timestamp < self.next_allowed.get(message, 0):
            return False                          # còn khóa: KHÔNG cập nhật
        self.next_allowed[message] = timestamp + 10
        return True
```

**Checklist**
- [ ] Tên class `Logger`, method `shouldPrintMessage(self, timestamp, message)` đúng chính tả?
- [ ] Trạng thái khởi tạo trong `__init__` qua `self.` (không dùng biến lớp dùng chung)?
- [ ] Chỉ cập nhật mốc thời gian khi **trả `True`**?
- [ ] Biên `t + 10` được in lại (dùng `<` chứ không phải `<=`)?
- [ ] Thông điệp mới luôn được in (mặc định đúng)?
- [ ] Nếu người phỏng vấn hỏi về bộ nhớ: nêu cách deque + set hoặc hai xô.